In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # comment this out if running interactively and you want inline plots
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Updated for Iris dataset
NUMERIC_COLS = [
    "SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"
]
CATEGORICAL_COLS = [
    "Species"
]

print("=" * 70)
print("IRIS DATASET CLUSTERING: K-Means / Hierarchical / DBSCAN")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Load + preprocess
# ---------------------------------------------------------------------------
# >>> EDIT the filename below so it exactly matches your CSV file <<<
df = pd.read_csv("/content/Iris (2) (3).csv")

imputer = SimpleImputer(strategy="median")
num_df = pd.DataFrame(
    imputer.fit_transform(df[NUMERIC_COLS]), columns=NUMERIC_COLS, index=df.index
)
cat_df = pd.get_dummies(df[CATEGORICAL_COLS], drop_first=True)

feature_df = pd.concat([num_df, cat_df], axis=1)
X = StandardScaler().fit_transform(feature_df)

print(f"Loaded {len(df):,} samples, {feature_df.shape[1]} clustering features")

IRIS DATASET CLUSTERING: K-Means / Hierarchical / DBSCAN
Loaded 150 samples, 6 clustering features


In [4]:
# ---------------------------------------------------------------------------
# 2. Choose k for K-Means via elbow + silhouette
# ---------------------------------------------------------------------------
k_range = range(2, 9)
# The original code tried to sample 5000 points from a population of 150, which caused a ValueError.
# Since the dataset is small (150 samples), we can use the full dataset for silhouette score calculation.
# If the dataset were larger, a smaller, representative sample would be appropriate.
X_sil_sample = X # Use the full dataset for silhouette score calculation

inertias, sil_scores = [], []
for k in k_range:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km_k.fit_predict(X)
    inertias.append(km_k.inertia_)
    sil_scores.append(silhouette_score(X_sil_sample, km_k.predict(X_sil_sample)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(list(k_range), inertias, marker="o")
ax1.set_title("Elbow method (inertia)")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")

ax2.plot(list(k_range), sil_scores, marker="o", color="darkorange")
ax2.set_title("Silhouette score vs k")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette score")

plt.tight_layout()
plt.savefig("kmeans_elbow_silhouette.png", dpi=150)
print("Saved plot -> kmeans_elbow_silhouette.png")
plt.close(fig)

best_k = list(k_range)[int(np.argmax(sil_scores))]
print(f"Silhouette-suggested k = {best_k}  (scores: "
      + ", ".join(f"k={k}:{s:.3f}" for k, s in zip(k_range, sil_scores)) + ")")

Saved plot -> kmeans_elbow_silhouette.png
Silhouette-suggested k = 3  (scores: k=2:0.505, k=3:0.639, k=4:0.556, k=5:0.470, k=6:0.401, k=7:0.387, k=8:0.389)


In [6]:
# ---------------------------------------------------------------------------
# 3. K-Means on the FULL dataset
# ---------------------------------------------------------------------------
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
km_labels = kmeans.fit_predict(X)

# Updated to use X_sil_sample (which is now the full X) and correct labels
sil = silhouette_score(X_sil_sample, km_labels)
print(f"\nK-Means (k={best_k}) silhouette (full dataset): {sil:.3f}")
print("Cluster sizes:", pd.Series(km_labels).value_counts().sort_index().to_dict())

# --- Profile clusters against OriginalSpecies and features ---
profiled = feature_df.copy()
profiled["KMeansCluster"] = km_labels
# Add the original 'Species' column for comparison
profiled["OriginalSpecies"] = df["Species"].values

summary = profiled.groupby("KMeansCluster").agg(
    n_samples=("OriginalSpecies", "size"),
    avg_SepalLengthCm=("SepalLengthCm", "mean"),
    avg_SepalWidthCm=("SepalWidthCm", "mean"),
    avg_PetalLengthCm=("PetalLengthCm", "mean"),
    avg_PetalWidthCm=("PetalWidthCm", "mean"),
    # Proportions of one-hot encoded species within each cluster
    prop_Iris_versicolor=("Species_Iris-versicolor", "mean"),
    prop_Iris_virginica=("Species_Iris-virginica", "mean"),
).round(3)

print("\n--- Cluster profiles (KMeansCluster) ---")
print(summary.to_string())
summary.to_csv("cluster_profiles.csv")
print("Saved table -> cluster_profiles.csv")


K-Means (k=3) silhouette (full dataset): 0.639
Cluster sizes: {0: 50, 1: 50, 2: 50}

--- Cluster profiles (KMeansCluster) ---
               n_samples  avg_SepalLengthCm  avg_SepalWidthCm  avg_PetalLengthCm  avg_PetalWidthCm  prop_Iris_versicolor  prop_Iris_virginica
KMeansCluster                                                                                                                                
0                     50              5.936             2.770              4.260             1.326                   1.0                  0.0
1                     50              6.588             2.974              5.552             2.026                   0.0                  1.0
2                     50              5.006             3.418              1.464             0.244                   0.0                  0.0
Saved table -> cluster_profiles.csv


In [8]:
# ---------------------------------------------------------------------------
# 4. Hierarchical + DBSCAN on the full dataset
# ---------------------------------------------------------------------------
# The previous attempt to subsample 5000 points from 150 caused a ValueError.
# For the Iris dataset (150 samples), we can use the full dataset without performance issues.
X_sub = X

hc = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hc_labels = hc.fit_predict(X_sub)
sil_hc = silhouette_score(X_sub, hc_labels)
print(f"\nHierarchical (k={best_k}, full dataset n={len(X_sub)}) silhouette: {sil_hc:.3f}")

db = DBSCAN(eps=4.0, min_samples=15)
db_labels = db.fit_predict(X_sub)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int(np.sum(db_labels == -1))
print(f"DBSCAN (full dataset n={len(X_sub)}): {n_db_clusters} clusters, {n_noise} noise points")
if n_db_clusters >= 2:
    mask = db_labels != -1
    print(f"DBSCAN silhouette (excl. noise): {silhouette_score(X_sub[mask], db_labels[mask]):.3f}")

# --- Dendrogram (subsample for readability, if dataset were larger) ---
# For the Iris dataset, use the full dataset for the dendrogram as it's small.
dendro_sample_size = min(len(X_sub), 80) # Limit for readability if dataset was slightly larger
dendro_idx = np.random.choice(len(X_sub), size=dendro_sample_size, replace=False)
Z = linkage(X_sub[dendro_idx], method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.title("Hierarchical Clustering Dendrogram (Ward linkage, full dataset)")
plt.xlabel("Sample index")
plt.ylabel("Distance")
plt.tight_layout()
plt.savefig("dendrogram.png", dpi=150)
print("Saved plot -> dendrogram.png")
plt.close()


Hierarchical (k=3, full dataset n=150) silhouette: 0.639
DBSCAN (full dataset n=150): 1 clusters, 0 noise points
Saved plot -> dendrogram.png


In [11]:
# ---------------------------------------------------------------------------
# 5. PCA for visualization
# ---------------------------------------------------------------------------
X_pca_full = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

# K-Means plot (full data)
fig, ax = plt.subplots(figsize=(6, 5.2))
ax.scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=km_labels, cmap="tab10", s=8, alpha=0.6)
ax.set_title(f"K-Means, full data ({best_k} clusters)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("kmeans_pca.png", dpi=150)
print("Saved plot -> kmeans_pca.png")
plt.close(fig)

# Hierarchical + DBSCAN plot (full data, not subsample anymore)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
# Use the full PCA-transformed data directly since X_sub is now X
coords_full = X_pca_full

axes[0].scatter(coords_full[:, 0], coords_full[:, 1], c=hc_labels, cmap="tab10", s=8, alpha=0.6)
axes[0].set_title(f"Hierarchical, full data ({best_k} clusters)")

noise_mask = db_labels == -1
axes[1].scatter(
    coords_full[~noise_mask, 0], coords_full[~noise_mask, 1],
    c=db_labels[~noise_mask], cmap="tab10", s=8, alpha=0.6,
)
axes[1].scatter(
    coords_full[noise_mask, 0], coords_full[noise_mask, 1], # Corrected coords_mask to coords_full
    c="lightgray", s=8, alpha=0.5, marker="x", label="noise",
)
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_title(f"DBSCAN, full data ({n_db_clusters} clusters)")

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.savefig("hierarchical_dbscan_pca.png", dpi=150)
print("Saved plot -> hierarchical_dbscan_pca.png")
plt.close(fig)

Saved plot -> kmeans_pca.png
Saved plot -> hierarchical_dbscan_pca.png


In [12]:
# ---------------------------------------------------------------------------
# 6. Save deliverable: original data + cluster assignment
# ---------------------------------------------------------------------------
df_out = df.copy()
df_out["KMeansCluster"] = km_labels
df_out.to_csv("placement_predict_with_clusters.csv", index=False)
print("\nSaved deliverable -> placement_predict_with_clusters.csv "
      "(original data + KMeansCluster column)")

print("\nDone. K-Means gives a clean, scalable segmentation of all students "
      "into profiles; the placement rate per cluster (see cluster_profiles.csv) "
      "shows which profiles need the most placement-cell attention.")



Saved deliverable -> placement_predict_with_clusters.csv (original data + KMeansCluster column)

Done. K-Means gives a clean, scalable segmentation of all students into profiles; the placement rate per cluster (see cluster_profiles.csv) shows which profiles need the most placement-cell attention.
